# Notebook 01 — Data Preprocessing & Exploratory Data Analysis

**COMP90049 Introduction to Machine Learning — Assignment 2**  
**Author (this notebook):** Yifan Wu  
**Dataset:** UK Used Car Dataset — https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes

---

## Overview

This notebook covers the full data preparation pipeline for our used car price prediction task:

1. **Data Loading** — merge per-brand CSV files into a single dataset
2. **Exploratory Data Analysis (EDA)** — understand distributions, correlations, and anomalies
3. **Data Cleaning** — handle duplicates, invalid values, and outliers
4. **Missing Value Handling** — analyse and impute missing entries
5. **Categorical Encoding** — convert text features to numeric representations
6. **Feature Engineering** — create informative derived features
7. **Feature Scaling** — prepare numerical features for distance-sensitive models
8. **Export** — save the cleaned dataset for downstream modelling notebooks

The processed dataset produced here is the single input used by all modelling notebooks (02–04), ensuring a consistent preprocessing baseline across all experiments.

## 1. Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

# Paths
DATA_DIR   = os.path.join('..', 'data')
OUTPUT_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Libraries loaded.')
print(f'Data directory : {os.path.abspath(DATA_DIR)}')
print(f'Output directory: {os.path.abspath(OUTPUT_DIR)}')

## 2. Data Loading

The dataset is distributed as one CSV per brand. We load all **clean** brand files and concatenate them into a single DataFrame, adding a `brand` column derived from the filename. The `unclean` variants are treated separately in Section 4 to illustrate preprocessing challenges.

**Clean files share a common schema:**
`model, year, price, transmission, mileage, fuelType, tax, mpg, engineSize`

In [ ]:
# Brand name mapping: filename stem -> display name
BRAND_MAP = {
    'audi'    : 'Audi',
    'bmw'     : 'BMW',
    'cclass'  : 'Mercedes',
    'focus'   : 'Ford',
    'ford'    : 'Ford',
    'hyundi'  : 'Hyundai',
    'merc'    : 'Mercedes',
    'skoda'   : 'Skoda',
    'toyota'  : 'Toyota',
    'vauxhall': 'Vauxhall',
    'vw'      : 'VW',
}

# Standard columns expected in every clean brand file
STANDARD_COLS = ['model', 'year', 'price', 'transmission', 'mileage',
                 'fuelType', 'tax', 'mpg', 'engineSize']

frames = []
for fname, brand in BRAND_MAP.items():
    fpath = os.path.join(DATA_DIR, f'{fname}.csv')
    if not os.path.exists(fpath):
        print(f'  [SKIP] {fpath} not found')
        continue
    df_tmp = pd.read_csv(fpath)
    df_tmp.columns = df_tmp.columns.str.strip()
    df_tmp['brand'] = brand
    # Keep only the standard schema — drops unexpected columns like 'tax(£)'
    keep = [c for c in STANDARD_COLS if c in df_tmp.columns] + ['brand']
    df_tmp = df_tmp[keep]
    frames.append(df_tmp)
    print(f'  Loaded {fname}.csv — {len(df_tmp):,} rows')

df = pd.concat(frames, ignore_index=True)
print(f'\nCombined dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
df.head(10)

In [ ]:
print('Column dtypes:')
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 3. Exploratory Data Analysis (EDA)

Before cleaning, we explore the raw data to understand its structure, distributions, and potential issues. This informs our cleaning and feature engineering decisions.

### 3.1 Basic Statistics

In [ ]:
df.describe(include='all').T

In [ ]:
# Missing value overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df if not missing_df.empty else 'None found in raw data.')

### 3.2 Target Variable: Price Distribution

Understanding the price distribution is essential — a heavily right-skewed target can hurt linear models and benefit from a log transformation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['price'].dropna(), bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Raw)')
axes[0].set_xlabel('Price (£)')
axes[0].set_ylabel('Count')

log_price = np.log1p(df['price'].dropna())
axes[1].hist(log_price, bins=80, color='darkorange', edgecolor='white')
axes[1].set_title('Price Distribution (log1p transformed)')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Count')

plt.suptitle('Figure 1: Raw vs. Log-transformed Price', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_price_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Price skewness (raw)     : {df['price'].skew():.3f}")
print(f"Price skewness (log1p)   : {log_price.skew():.3f}")

### 3.3 Sample Size per Brand

In [ ]:
brand_counts = df['brand'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
brand_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Figure 2: Number of Listings per Brand')
ax.set_xlabel('Brand')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_brand_counts.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Price by Brand

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
brand_order = df.groupby('brand')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='brand', y='price', order=brand_order, ax=ax,
            palette='Set2', showfliers=False)
ax.set_title('Figure 3: Price Distribution by Brand (outliers hidden for clarity)')
ax.set_xlabel('Brand')
ax.set_ylabel('Price (£)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_price_by_brand.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Categorical Feature Distributions

In [ ]:
cat_cols = ['transmission', 'fuelType']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts()
    vc.plot(kind='bar', ax=ax, color='teal', edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30)

plt.suptitle('Figure 4: Categorical Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig4_cat_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.6 Numerical Feature Distributions

In [ ]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=60, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

axes[-1].set_visible(False)
plt.suptitle('Figure 5: Numerical Feature Distributions (Raw)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig5_num_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.7 Correlation with Price

In [ ]:
corr_cols = ['price', 'year', 'mileage', 'tax', 'mpg', 'engineSize']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Figure 6: Pearson Correlation Matrix (Numerical Features)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig6_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Price vs mileage and year — the two strongest correlates
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sample = df.sample(min(5000, len(df)), random_state=42)

axes[0].scatter(sample['mileage'], sample['price'], alpha=0.3, s=10, color='steelblue')
axes[0].set_title('Price vs Mileage')
axes[0].set_xlabel('Mileage')
axes[0].set_ylabel('Price (£)')

axes[1].scatter(sample['year'], sample['price'], alpha=0.3, s=10, color='darkorange')
axes[1].set_title('Price vs Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Price (£)')

plt.suptitle('Figure 7: Price vs Key Numerical Features', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig7_price_vs_features.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Cleaning

We identify and address four categories of data quality issues:
- Duplicate rows
- Invalid or physically impossible values (e.g., `engineSize = 0`, `mileage = 0` on a used car)
- Extreme outliers in price, mileage, and tax
- Rare / inconsistent categories in categorical columns

We also briefly demonstrate the challenges present in the **unclean** files to motivate why thorough cleaning matters.

### 4.1 Demonstrating the Unclean Data Challenge

The `unclean focus.csv` and `unclean cclass.csv` files illustrate real-world messy data issues: price stored as a formatted British pound string (`£8,000`), mileage spread across two columns, and extra reference columns. This is shown for context; our main pipeline uses the pre-cleaned per-brand files.

In [ ]:
df_dirty = pd.read_csv(os.path.join(DATA_DIR, 'unclean focus.csv'))
df_dirty.columns = df_dirty.columns.str.strip()
print('Unclean Focus — shape:', df_dirty.shape)
df_dirty.head()

In [ ]:
# Issues in the unclean file:
print('1. Price column (raw):', df_dirty['price'].head(5).tolist())
print('\n2. Mileage column is empty; real mileage is in mileage2:')
print(df_dirty[['mileage', 'mileage2']].head())
print('\n3. Extra columns:', [c for c in df_dirty.columns if c not in
      ['model','year','price','transmission','mileage','fuel type','engine size']])

In [ ]:
def clean_unclean_file(path, brand_name):
    """Parse a messy brand CSV into the standard schema."""
    d = pd.read_csv(path)
    d.columns = d.columns.str.strip()

    # Fix price: strip '£' / commas, coerce non-numeric to NaN
    d['price'] = pd.to_numeric(
        d['price'].astype(str)
                  .str.replace('£', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .str.strip(),
        errors='coerce'
    )

    # mileage2 holds the real mileage; 'Unknown' and other non-numeric → NaN
    d['mileage'] = pd.to_numeric(
        d['mileage2'].astype(str)
                     .str.replace(',', '', regex=False)
                     .str.strip(),
        errors='coerce'
    )

    # Standardise column names to match clean files
    d = d.rename(columns={'fuel type': 'fuelType', 'engine size': 'engineSize'})

    # Drop redundant / unneeded columns
    keep = ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'engineSize']
    d = d[[c for c in keep if c in d.columns]]

    # Add missing columns with NaN so schema aligns with main dataset
    for col in ['tax', 'mpg']:
        if col not in d.columns:
            d[col] = np.nan

    d['brand'] = brand_name
    return d

df_clean_focus  = clean_unclean_file(os.path.join(DATA_DIR, 'unclean focus.csv'),  'Ford')
df_clean_cclass = clean_unclean_file(os.path.join(DATA_DIR, 'unclean cclass.csv'), 'Mercedes')

print('Cleaned unclean Focus  :', df_clean_focus.shape)
print('Cleaned unclean C-Class:', df_clean_cclass.shape)
df_clean_focus.head(3)

### 4.2 Duplicate Removal

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Removed {before - after:,} duplicate rows ({(before-after)/before*100:.2f}%)')
print(f'Dataset size after dedup: {after:,} rows')

### 4.3 Invalid Values

Several values are physically impossible for a used car listing and treated as missing:
- `engineSize == 0`: likely data entry error (most petrol/diesel engines are > 0.5L)
- `mileage == 0`: a used car listing with zero mileage is suspicious (could be a placeholder)
- `price == 0` or `price < 100`: unrealistic for a car listing
- `year < 1990` or `year > 2023`: outside the realistic range of second-hand cars in this context

In [ ]:
print('Value counts for suspect conditions:')
print(f"  engineSize == 0 : {(df['engineSize'] == 0).sum():,}")
print(f"  mileage   == 0  : {(df['mileage'] == 0).sum():,}")
print(f"  price     < 100 : {(df['price'] < 100).sum():,}")
print(f"  year      < 1990: {(df['year'] < 1990).sum():,}")
print(f"  year      > 2023: {(df['year'] > 2023).sum():,}")

In [ ]:
# Replace invalid values with NaN (to be handled in the imputation step)
df.loc[df['engineSize'] == 0, 'engineSize'] = np.nan
df.loc[df['mileage'] == 0, 'mileage'] = np.nan
df.loc[df['price'] < 100, 'price'] = np.nan
df = df[df['year'].between(1990, 2023)].copy()

print(f'Dataset size after invalid value handling: {len(df):,} rows')

### 4.4 Rare / Inconsistent Categories

In [ ]:
print('transmission value counts:')
print(df['transmission'].value_counts())
print('\nfuelType value counts:')
print(df['fuelType'].value_counts())

In [ ]:
# Keep only the four main fuel types; map rare ones to NaN
valid_fuel = {'Petrol', 'Diesel', 'Hybrid', 'Electric'}
df.loc[~df['fuelType'].isin(valid_fuel), 'fuelType'] = np.nan

# Keep only the three main transmission types
valid_trans = {'Manual', 'Automatic', 'Semi-Auto'}
df.loc[~df['transmission'].isin(valid_trans), 'transmission'] = np.nan

print('After cleaning categories:')
print('  transmission:', df['transmission'].value_counts().to_dict())
print('  fuelType    :', df['fuelType'].value_counts().to_dict())

### 4.5 Outlier Detection and Removal

We use the **IQR (Interquartile Range) method** to detect and remove extreme outliers in `price`, `mileage`, and `tax`. Points below Q1 − 3×IQR or above Q3 + 3×IQR are removed (a conservative threshold to retain genuine high/low-value cars while removing data entry errors).

In [ ]:
def remove_outliers_iqr(df, col, factor=3.0):
    """Remove rows where col is beyond factor * IQR from Q1/Q3."""
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    before = len(df)
    df = df[df[col].between(lower, upper) | df[col].isna()].copy()
    print(f'  [{col}] bounds: ({lower:.0f}, {upper:.0f}) — removed {before - len(df):,} rows')
    return df

print('Outlier removal (IQR × 3):')
df = remove_outliers_iqr(df, 'price')
df = remove_outliers_iqr(df, 'mileage')
df = remove_outliers_iqr(df, 'tax')
print(f'\nDataset size after outlier removal: {len(df):,} rows')

## 5. Missing Value Analysis & Handling

After cleaning, we assess the remaining missing values and apply appropriate imputation strategies. We compare three strategies to support **Research Question 3** (impact of preprocessing on model performance).

In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %'    : (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Missing values after cleaning:')
print(missing_summary)

In [ ]:
# Visualise missingness pattern
if not missing_summary.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_summary['Missing %'].plot(kind='barh', ax=ax, color='salmon', edgecolor='white')
    ax.set_title('Figure 8: Missing Value Percentage by Feature')
    ax.set_xlabel('Missing %')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig8_missing_values.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing values — skip chart.')

### 5.1 Imputation Strategies

| Feature | Strategy | Rationale |
|---|---|---|
| `engineSize` | **Median** within `(brand, fuelType)` group | Engine size is brand- and fuel-specific |
| `tax` | **Median** within `(year, engineSize_bin)` group | Tax is tied to CO₂ emissions, which correlate with year and engine size |
| `mpg` | **Median** within `(fuelType, engineSize_bin)` group | MPG depends on powertrain type |
| `mileage` | **Median** within `(year, brand)` group | Expected mileage scales with age and brand usage patterns |
| `fuelType`, `transmission` | **Mode** (most frequent within brand) | Categorical — mode is the least-assumptions imputation |

In [ ]:
def group_median_impute(df, target_col, group_cols):
    """Fill NaN in target_col with the median of its group."""
    group_medians = df.groupby(group_cols)[target_col].transform('median')
    overall_median = df[target_col].median()
    filled = df[target_col].fillna(group_medians).fillna(overall_median)
    n_filled = df[target_col].isna().sum()
    print(f'  {target_col}: filled {n_filled:,} missing values (group median by {group_cols})')
    return filled

# Create a coarse engine size bin for grouping
df['engineSize_bin'] = pd.cut(df['engineSize'].fillna(df['engineSize'].median()),
                               bins=[0, 1.0, 1.5, 2.0, 3.0, 10.0],
                               labels=['≤1.0','1.0-1.5','1.5-2.0','2.0-3.0','>3.0'])

print('Imputing numerical columns:')
df['engineSize'] = group_median_impute(df, 'engineSize', ['brand', 'fuelType'])
df['tax']        = group_median_impute(df, 'tax',        ['year', 'engineSize_bin'])
df['mpg']        = group_median_impute(df, 'mpg',        ['fuelType', 'engineSize_bin'])
df['mileage']    = group_median_impute(df, 'mileage',    ['year', 'brand'])
df['price']      = df['price'].fillna(df['price'].median())  # fallback for price

print('\nImputing categorical columns:')
for col in ['fuelType', 'transmission']:
    mode_map = df.groupby('brand')[col].agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
    df[col] = df.apply(lambda row: mode_map[row['brand']] if pd.isna(row[col]) else row[col], axis=1)
    print(f'  {col}: remaining NaN = {df[col].isna().sum()}')

df.drop(columns=['engineSize_bin'], inplace=True)

print(f'\nTotal remaining NaN: {df.isnull().sum().sum()}')

## 6. Categorical Encoding

Machine learning models require numerical inputs. We apply two encoding strategies:

- **Label Encoding** for `transmission` — it has a natural ordinal-ish structure (Manual < Semi-Auto < Automatic in terms of automation level) and only 3 levels.
- **One-Hot Encoding (OHE)** for `fuelType` — 4 categories with no intrinsic order; OHE avoids imposing false ordinality.
- **Target Encoding** for `brand` and `model` — these are high-cardinality categoricals. Target encoding replaces each category with the mean target value (log price), which is more compact than OHE and often more informative for tree-based models.

We save the mapping dictionaries so that the test set can be encoded consistently.

In [ ]:
# --- Label encoding: transmission ---
trans_order = {'Manual': 0, 'Semi-Auto': 1, 'Automatic': 2}
df['transmission_enc'] = df['transmission'].map(trans_order)
print('transmission label encoding:', trans_order)

In [ ]:
# --- One-hot encoding: fuelType ---
fuel_dummies = pd.get_dummies(df['fuelType'], prefix='fuel', drop_first=False)
df = pd.concat([df, fuel_dummies], axis=1)
print('OHE columns added:', fuel_dummies.columns.tolist())

In [ ]:
# --- Target encoding: brand and model ---
# We compute target encoding on the FULL dataset here (for EDA purposes).
# In the modelling notebooks, this is fitted on train only to prevent leakage.
log_price = np.log1p(df['price'])

brand_target_enc = df.groupby('brand')['price'].mean().rename('brand_mean_price')
model_target_enc = df.groupby('model')['price'].mean().rename('model_mean_price')

df['brand_mean_price'] = df['brand'].map(brand_target_enc)
df['model_mean_price'] = df['model'].map(model_target_enc)

print('Brand target encoding (top 5):')
print(brand_target_enc.sort_values(ascending=False).head())
print('\nSample model target encoding:')
print(model_target_enc.head())

## 7. Feature Engineering

Beyond the raw features, we construct several derived features that encode domain knowledge about the used car market:

| New Feature | Formula | Rationale |
|---|---|---|
| `car_age` | `2024 − year` | Depreciation is primarily a function of age |
| `mileage_per_year` | `mileage / (car_age + 1)` | Normalises mileage by age; a low-mileage old car behaves differently from a high-mileage new car |
| `is_luxury` | `1` if brand in {BMW, Audi, Mercedes} | Premium brands command a price premium independent of other features |
| `log_price` | `log1p(price)` | Log-transformed target; reduces right-skew and stabilises variance for regression models |

In [ ]:
CURRENT_YEAR = 2024
LUXURY_BRANDS = {'BMW', 'Audi', 'Mercedes'}

df['car_age']         = CURRENT_YEAR - df['year']
df['mileage_per_year']= df['mileage'] / (df['car_age'] + 1)
df['is_luxury']       = df['brand'].isin(LUXURY_BRANDS).astype(int)
df['log_price']       = np.log1p(df['price'])

print('Engineered features:')
print(df[['car_age', 'mileage_per_year', 'is_luxury', 'log_price']].describe())

In [ ]:
# Visualise car_age vs log_price to confirm the relationship is cleaner than year vs price
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sample = df.sample(min(5000, len(df)), random_state=42)

axes[0].scatter(sample['car_age'], sample['log_price'], alpha=0.3, s=10, color='teal')
axes[0].set_title('log(Price) vs Car Age')
axes[0].set_xlabel('Car Age (years)')
axes[0].set_ylabel('log(Price + 1)')

axes[1].scatter(sample['mileage_per_year'], sample['log_price'], alpha=0.3, s=10, color='purple')
axes[1].set_title('log(Price) vs Mileage per Year')
axes[1].set_xlabel('Mileage per Year')
axes[1].set_ylabel('log(Price + 1)')

plt.suptitle('Figure 9: Engineered Features vs Log Price', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig9_engineered_features.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Scaling

Scaling is particularly important for distance-based models (KNN) and gradient-based models (Neural Networks), which are sensitive to feature magnitude differences. Linear Regression with regularisation also benefits from standardised inputs.

We prepare **two scaled versions** for comparison in RQ3:
- `StandardScaler` (zero mean, unit variance) — standard choice for most models
- `MinMaxScaler` ([0,1] range) — preferred when bounded inputs are expected

Tree-based models (Random Forest, Gradient Boosting) are scale-invariant and will use the raw feature values.

In [ ]:
# Define the final feature set for modelling
FEATURE_COLS = [
    'car_age', 'mileage', 'mileage_per_year',
    'tax', 'mpg', 'engineSize',
    'transmission_enc', 'is_luxury',
    'brand_mean_price', 'model_mean_price',
    'fuel_Diesel', 'fuel_Electric', 'fuel_Hybrid', 'fuel_Petrol',
]

# Check which columns exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
TARGET_COL   = 'log_price'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape        : {y.shape}')
print(f'Features used       : {FEATURE_COLS}')

In [ ]:
# Apply StandardScaler
scaler_std = StandardScaler()
X_std = pd.DataFrame(scaler_std.fit_transform(X), columns=FEATURE_COLS, index=X.index)

# Apply MinMaxScaler
scaler_mm = MinMaxScaler()
X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=FEATURE_COLS, index=X.index)

print('StandardScaler — feature means (should be ~0):')
print(X_std.mean().round(4))
print('\nMinMaxScaler — feature ranges (should be [0, 1]):')
print(pd.DataFrame({'min': X_mm.min(), 'max': X_mm.max()}))

In [ ]:
# Visualise effect of scaling on a subset of features
compare_cols = ['car_age', 'mileage', 'engineSize', 'mpg']
fig, axes = plt.subplots(3, len(compare_cols), figsize=(16, 9))
titles = ['Raw', 'StandardScaler', 'MinMaxScaler']
datasets = [X, X_std, X_mm]

for row, (title, data) in enumerate(zip(titles, datasets)):
    for col_i, col in enumerate(compare_cols):
        if col in data.columns:
            axes[row, col_i].hist(data[col].dropna(), bins=50, color='steelblue', edgecolor='white')
            if row == 0:
                axes[row, col_i].set_title(col)
            if col_i == 0:
                axes[row, col_i].set_ylabel(title, fontsize=10)

plt.suptitle('Figure 10: Effect of Feature Scaling on Numerical Distributions', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig10_scaling_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Dataset Summary

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Total rows            : {len(df):,}')
print(f'Total features (raw)  : {df.shape[1]}')
print(f'Modelling features    : {len(FEATURE_COLS)}')
print(f'Target                : {TARGET_COL} (log1p of price in £)')
print(f'Remaining NaN         : {df[FEATURE_COLS].isnull().sum().sum()}')
print()
print('Brand distribution in final dataset:')
print(df['brand'].value_counts())
print()
print('Price summary (original scale):')
print(df['price'].describe())

In [ ]:
# Final pairplot of key features coloured by luxury status
plot_cols = ['car_age', 'mileage_per_year', 'engineSize', 'log_price', 'is_luxury']
plot_sample = df[plot_cols].sample(min(3000, len(df)), random_state=42)

g = sns.pairplot(plot_sample, hue='is_luxury', plot_kws={'alpha': 0.4, 's': 15},
                 palette={0: 'steelblue', 1: 'darkorange'},
                 vars=['car_age', 'mileage_per_year', 'engineSize', 'log_price'])
g.fig.suptitle('Figure 11: Pairplot of Key Features (orange = luxury brand)', y=1.02, fontsize=13)
g.fig.savefig(os.path.join(OUTPUT_DIR, 'fig11_pairplot.png'), dpi=120, bbox_inches='tight')
plt.show()

## 10. Export Processed Dataset

We export the cleaned and feature-engineered dataset. The modelling notebooks (02–04) load this file as their starting point, ensuring consistent preprocessing across all experiments.

In [ ]:
# Assemble final output — keep raw price alongside log_price for interpretability
output_cols = FEATURE_COLS + ['price', 'log_price', 'brand', 'model', 'transmission', 'fuelType', 'year']
output_cols = [c for c in output_cols if c in df.columns]
df_out = df[output_cols].copy()

out_path = os.path.join(OUTPUT_DIR, 'cars_cleaned.csv')
df_out.to_csv(out_path, index=False)

print(f'Saved cleaned dataset to: {out_path}')
print(f'Shape: {df_out.shape}')
df_out.head()

---

## Summary

| Step | Action | Outcome |
|---|---|---|
| Loading | Merged 11 brand CSVs | ~108k rows, 9 raw features |
| Deduplication | Removed exact duplicates | Minimal reduction |
| Invalid values | Zeroed engineSize/mileage → NaN; filtered year | Dataset integrity improved |
| Outliers | IQR × 3 on price, mileage, tax | Removed extreme data errors |
| Imputation | Group-median (numerical), mode (categorical) | Zero remaining NaN |
| Encoding | Label (transmission), OHE (fuelType), Target (brand, model) | All features numeric |
| Feature engineering | `car_age`, `mileage_per_year`, `is_luxury`, `log_price` | 4 informative derived features |
| Scaling | StandardScaler and MinMaxScaler prepared | Ready for distance-sensitive models |
| Export | `data/processed/cars_cleaned.csv` | Input for modelling notebooks |

The cleaned dataset is now ready for **Notebook 02** (baseline models: Linear Regression, KNN).